In [1]:
!pip -q install sgp4 skyfield numpy pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.1/218.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.1/371.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.8 MB/s eta 0:00:00


Mission Control: ISS Ground Track & Orbital Analysis

This project develops a satellite-tracking and orbital-analysis workflow using real International Space Station (ISS) orbital data. It combines SGP4 propagation, geographic coordinate conversion, interactive visualization, and telemetry analysis to explore satellite motion in both space and over Earth's surface.

The workflow uses Python, Skyfield, SGP4, NumPy, Pandas, and Plotly.

1. Environment Setup and Dependencies

The required scientific computing and visualization libraries are installed to build the orbital-analysis pipeline. SGP4 provides satellite orbit propagation, Skyfield handles time and geographic coordinate calculations, NumPy supports numerical operations, Pandas organizes telemetry data, and Plotly creates interactive visualizations.

In [3]:
import requests
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd

from skyfield.api import EarthSatellite, load, wgs84
from sgp4.api import Satrec, jday

TLE_URL = (
    "https://celestrak.org/NORAD/elements/gp.php"
    "?CATNR=25544&FORMAT=TLE"
)

response = requests.get(TLE_URL, timeout=30)
response.raise_for_status()

lines = [line.strip() for line in response.text.splitlines()
         if line.strip()]

if len(lines) < 3:
    raise ValueError("TLE verisi eksik geldi. Hücreyi tekrar çalıştır.")

satellite_name = lines[0]
tle1, tle2 = lines[1], lines[2]

# twoline2satrec yerine doğru API olan twoline2rv kullanılıyor
satrec = Satrec.twoline2rv(tle1, tle2)

# Skyfield, TLE'yi zaman ve koordinat hesaplarında kullanacak.
ts = load.timescale()
satellite = EarthSatellite(tle1, tle2, satellite_name, ts)

now = datetime.now(timezone.utc)

print("Uydu:", satellite_name)
print("NORAD ID: 25544 (ISS)")
print("TLE 1:", tle1)
print("TLE 2:", tle2)
print("Hesaplama zamanı UTC:", now.isoformat())

Uydu: ISS (ZARYA)
NORAD ID: 25544 (ISS)
TLE 1: 1 25544U 98067A   26285.14422154  .00005771  00000+0  11345-3 0  9992
TLE 2: 2 25544  51.6314  81.3064 0006731 252.4466 107.5787 15.48833724589888
Hesaplama zamanı UTC: 2026-10-12T11:29:04.771074+00:00


2. Retrieving Real ISS Orbital Elements

Two-Line Element (TLE) data for the International Space Station is retrieved from CelesTrak using an HTTP request. The orbital elements are parsed and initialized for subsequent propagation with the SGP4 model.

TLE data describes the satellite's orbit at a reference epoch; it is not direct onboard sensor telemetry. The retrieved elements are used to estimate the ISS position and velocity over time.

In [5]:
# Define coordinates for Istanbul
# (You can modify the latitude and longitude to match your specific observer location)
istanbul = wgs84.latlon(41.0082, 28.9784)

# Define the time range (Next 24 hours starting from now)
t_start = ts.from_datetime(now)
t_end = ts.from_datetime(now + timedelta(hours=24))

# Find the ISS pass events relative to Istanbul
# (We search for passes where the satellite is at least 10 degrees above the horizon)
t, events = satellite.find_events(istanbul, t_start, t_end, altitude_degrees=10.0)

pass_events = []
current_pass = {}

event_names = ['Rise', 'Culmination', 'Set']

for ti, event in zip(t, events):
    name = event_names[event]
    g = (satellite - istanbul).at(ti)
    alt, az, distance = g.altaz()

    # Convert time to UTC and UTC+3 Local time for easy reference
    utc_time = ti.utc_datetime()
    local_time = utc_time + timedelta(hours=3)

    if event == 0:  # Rise
        current_pass = {
            'Start_UTC': utc_time.strftime('%H:%M:%S'),
            'Start_Local': local_time.strftime('%H:%M:%S'),
            'Start_Azimuth': f"{az.degrees:.1f}°"
        }
    elif event == 1:  # Culmination
        current_pass['Max_Altitude_Time'] = local_time.strftime('%H:%M:%S')
        current_pass['Max_Altitude'] = f"{alt.degrees:.1f}°"
        current_pass['Max_Altitude_Direction'] = f"{az.degrees:.1f}°"
    elif event == 2:  # Set
        if current_pass:
            current_pass['End_Local'] = local_time.strftime('%H:%M:%S')
            current_pass['End_Azimuth'] = f"{az.degrees:.1f}°"
            current_pass['Date'] = local_time.strftime('%Y-%m-%d')
            pass_events.append(current_pass)
            current_pass = {}

# Format the results into a pandas DataFrame and display it
if pass_events:
    df_passes = pd.DataFrame(pass_events)
    # Reorder columns for better presentation
    cols = ['Date', 'Start_Local', 'Max_Altitude_Time', 'End_Local', 'Max_Altitude', 'Start_Azimuth', 'End_Azimuth']
    df_passes = df_passes[cols]
    print("ISS Pass Schedule for Istanbul over the Next 24 Hours (Local Time UTC+3):")
    display(df_passes)
else:
    print("There are no visible ISS passes above 10 degrees elevation over Istanbul in the next 24 hours.")

ISS Pass Schedule for Istanbul over the Next 24 Hours (Local Time UTC+3):


,Date,Start_Local,Max_Altitude_Time,End_Local,Max_Altitude,Start_Azimuth,End_Azimuth
0,2026-10-12,14:42:27,14:45:36,14:48:43,35.9°,291.2°,155.2°
1,2026-10-13,07:25:14,07:28:37,07:32:01,79.9°,229.1°,55.9°
2,2026-10-13,09:03:04,09:05:46,09:08:28,21.3°,288.2°,33.1°
3,2026-10-13,10:41:43,10:43:33,10:45:24,13.7°,328.2°,33.9°
4,2026-10-13,12:18:29,12:21:18,12:24:07,23.5°,325.3°,77.3°
5,2026-10-13,13:55:00,13:58:22,14:01:43,63.5°,300.7°,137.7°


3. Propagating the Orbit and Computing Ground Coordinates

The ISS trajectory is propagated over a 90-minute interval using 30-second sampling steps. Skyfield computes the satellite's geographic subpoint, latitude, longitude, and altitude relative to the WGS84 reference ellipsoid.

SGP4 position and velocity outputs are also recorded for further analysis. The resulting dataset provides a time-resolved representation of the satellite's motion and its corresponding ground track.

In [7]:
# 90 minutes, sampling every 30 seconds
sample_times = [
    now + timedelta(seconds=30 * i)
    for i in range(181)
]

# Convert to Skyfield time objects
sky_times = ts.from_datetimes(sample_times)

# Calculate geocentric position and ground track for each sample
geocentric = satellite.at(sky_times)
subpoints = wgs84.subpoint_of(geocentric)

latitudes = subpoints.latitude.degrees
longitudes = subpoints.longitude.degrees
altitudes = wgs84.height_of(geocentric).km

# Velocity vectors are calculated directly from SGP4 output
# rather than the derivative of the GCRS position.
sgp4_positions = []
sgp4_velocities = []

for t in sample_times:
    jd, fr = jday(
        t.year, t.month, t.day,
        t.hour, t.minute,
        t.second + t.microsecond / 1e6
    )
    error, p, v = satrec.sgp4(jd, fr)
    if error != 0:
        raise RuntimeError(f"SGP4 error code: {error}")
    sgp4_positions.append(p)
    sgp4_velocities.append(v)

sgp4_positions = np.asarray(sgp4_positions)
sgp4_velocities = np.asarray(sgp4_velocities)
speeds = np.linalg.norm(sgp4_velocities, axis=1)

# Create telemetry DataFrame with English column headers
telemetry = pd.DataFrame({
    "time_utc": sample_times,
    "latitude_deg": latitudes,
    "longitude_deg": longitudes,
    "altitude_km": altitudes,
    "speed_km_s": speeds,
    "x_teme_km": sgp4_positions[:, 0],
    "y_teme_km": sgp4_positions[:, 1],
    "z_teme_km": sgp4_positions[:, 2],
})

display(telemetry.head())
print("Total samples:", len(telemetry))
print("Latitude range:", round(telemetry.latitude_deg.min(), 2),
      "to", round(telemetry.latitude_deg.max(), 2))

,time_utc,latitude_deg,longitude_deg,altitude_km,speed_km_s,x_teme_km,y_teme_km,z_teme_km
0,2026-10-12 11:29:04.771074+00:00,45.566175,-60.256085,429.289966,7.657214,-3259.758873,3487.046024,4838.159828
1,2026-10-12 11:29:34.771074+00:00,46.444592,-57.904703,429.508147,7.657169,-3353.989500,3290.521815,4910.706666
2,2026-10-12 11:30:04.771074+00:00,47.267615,-55.475157,429.709450,7.657131,-3444.396713,3090.246548,4977.639457
3,2026-10-12 11:30:34.771074+00:00,48.031904,-52.968227,429.892764,7.657099,-3530.877670,2886.448729,5038.882001
4,2026-10-12 11:31:04.771074+00:00,48.734124,-50.385685,430.057057,7.657074,-3613.333996,2679.360832,5094.364581


Total samples: 181
Latitude range: -51.78 to 51.78


4. Visualizing the ISS Ground Track

An interactive world map is generated with Plotly to display the ISS ground track over the 90-minute prediction window. Geographic coordinates are plotted as longitude-latitude pairs, with markers indicating the beginning and end of the trajectory.

Longitude discontinuities near the ±180° meridian are handled to prevent incorrect lines from crossing the map. This visualization illustrates how an orbital trajectory projects onto Earth's surface.

In [9]:
import plotly.graph_objects as go

# Prevent line wrapping across the map edge at the 180° longitude meridian crossing
map_lons = telemetry["longitude_deg"].to_numpy().copy()
map_lats = telemetry["latitude_deg"].to_numpy().copy()

jumps = np.where(np.abs(np.diff(map_lons)) > 180)[0]
map_lons[jumps + 1] = np.nan
map_lats[jumps + 1] = np.nan

fig_map = go.Figure()

fig_map.add_trace(go.Scattergeo(
    lon=map_lons,
    lat=map_lats,
    mode="lines",
    line={"width": 3, "color": "#00cfff"},
    name="ISS ground track",
    hovertemplate="Lat: %{lat:.2f}°<br>Lon: %{lon:.2f}°<extra></extra>"
))

fig_map.add_trace(go.Scattergeo(
    lon=[telemetry.longitude_deg.iloc[0]],
    lat=[telemetry.latitude_deg.iloc[0]],
    mode="markers+text",
    marker={"size": 10, "color": "#39ff88"},
    text=["START"],
    textposition="top center",
    name="Start"
))

fig_map.add_trace(go.Scattergeo(
    lon=[telemetry.longitude_deg.iloc[-1]],
    lat=[telemetry.latitude_deg.iloc[-1]],
    mode="markers+text",
    marker={"size": 10, "color": "#ffb347"},
    text=["+90 MIN"],
    textposition="top center",
    name="End"
))

fig_map.update_layout(
    title="MISSION CONTROL — ISS GROUND TRACK (90 MIN)",
    paper_bgcolor="#03101e",
    font={"color": "#d7eaff"},
    geo={
        "projection_type": "natural earth",
        "showland": True,
        "landcolor": "#18354a",
        "showocean": True,
        "oceancolor": "#061626",
        "showcoastlines": True,
        "coastlinecolor": "#3a7799",
        "showframe": False,
        "bgcolor": "#03101e",
        "lonaxis": {"showgrid": True, "gridcolor": "#17334d"},
        "lataxis": {"showgrid": True, "gridcolor": "#17334d"}
    },
    height=600
)

fig_map.show()

5. Orbital Telemetry Analysis

Altitude and speed are plotted against elapsed time to examine changes along the predicted trajectory. The altitude series is derived from Skyfield's WGS84 geographic calculations, while speed is computed as the magnitude of the velocity vector returned by SGP4.

These time-series plots provide a compact view of orbital behavior and help identify variations across the propagation interval.

In [10]:
from plotly.subplots import make_subplots

elapsed_minutes = np.arange(len(telemetry)) * 0.5

fig_analysis = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Altitude above WGS84 ellipsoid",
        "Speed magnitude from SGP4"
    )
)

fig_analysis.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=telemetry["altitude_km"],
        mode="lines",
        line={"color": "#35d7ff", "width": 2.5},
        name="Altitude"
    ),
    row=1, col=1
)

fig_analysis.add_trace(
    go.Scatter(
        x=elapsed_minutes,
        y=telemetry["speed_km_s"],
        mode="lines",
        line={"color": "#47f5a1", "width": 2.5},
        name="Speed"
    ),
    row=2, col=1
)

fig_analysis.update_layout(
    title="ISS ORBITAL TELEMETRY ANALYSIS",
    height=650,
    paper_bgcolor="#03101e",
    plot_bgcolor="#061626",
    font={"color": "#d7eaff"},
    legend={"orientation": "h", "y": 1.08}
)

fig_analysis.update_xaxes(
    title_text="Minutes from start",
    gridcolor="#18354a"
)
fig_analysis.update_yaxes(
    title_text="Altitude (km)",
    gridcolor="#18354a",
    row=1, col=1
)
fig_analysis.update_yaxes(
    title_text="Speed (km/s)",
    gridcolor="#18354a",
    row=2, col=1
)

fig_analysis.show()

6. Three-Dimensional Orbital Visualization

A 3D scene is constructed using Plotly to represent Earth as a sphere and the predicted ISS trajectory as a spatial curve. The satellite's initial position is highlighted to provide a reference point for exploring the orbit.

The scene supports interactive rotation and zooming. The Earth model is simplified for visualization, and its axes and orientation are not precisely aligned with the TEME reference frame used by the propagated orbital coordinates.

In [12]:
# 3D Orbit Visualization with TEME Positions.
# This view is for visualization purposes; the orientation of the Earth texture
# is not precisely aligned to the TEME frame.

import plotly.graph_objects as go
import numpy as np

R_EARTH = 6378.137
theta = np.linspace(0, 2 * np.pi, 90)
phi = np.linspace(0, np.pi, 45)

earth_x = R_EARTH * np.outer(np.cos(theta), np.sin(phi))
earth_y = R_EARTH * np.outer(np.sin(theta), np.sin(phi))
earth_z = R_EARTH * np.outer(np.ones_like(theta), np.cos(phi))

fig_3d = go.Figure()

# Earth surface layer
fig_3d.add_trace(go.Surface(
    x=earth_x,
    y=earth_y,
    z=earth_z,
    colorscale=[
        [0, "#071a33"],
        [0.45, "#075b91"],
        [0.75, "#138a8a"],
        [1, "#b7d8c8"]
    ],
    showscale=False,
    opacity=0.96,
    name="Earth",
    hoverinfo="skip"
))

orbit_positions = sgp4_positions

# Predicted orbit path
fig_3d.add_trace(go.Scatter3d(
    x=orbit_positions[:, 0],
    y=orbit_positions[:, 1],
    z=orbit_positions[:, 2],
    mode="lines",
    line={"color": "#32d8ff", "width": 5},
    name="Predicted Orbit",
    hovertemplate=(
        "X: %{x:.1f} km<br>"
        "Y: %{y:.1f} km<br>"
        "Z: %{z:.1f} km<extra>Orbit</extra>"
    )
))

# Initial ISS Position marker
fig_3d.add_trace(go.Scatter3d(
    x=[orbit_positions[0, 0]],
    y=[orbit_positions[0, 1]],
    z=[orbit_positions[0, 2]],
    mode="markers+text",
    marker={"size": 6, "color": "#47ff9b"},
    text=["ISS"],
    textposition="top center",
    name="Initial Position"
))

fig_3d.update_layout(
    title="MISSION CONTROL — 3D ORBITAL VIEW",
    height=750,
    paper_bgcolor="#020b16",
    font={"color": "#d7eaff"},
    scene={
        "bgcolor": "#020b16",
        "aspectmode": "data",
        "xaxis": {"title": "X (km)", "gridcolor": "#18354a"},
        "yaxis": {"title": "Y (km)", "gridcolor": "#18354a"},
        "zaxis": {"title": "Z (km)", "gridcolor": "#18354a"},
        "camera": {"eye": {"x": 1.6, "y": 1.6, "z": 1.2}}
    },
    margin={"l": 0, "r": 0, "t": 60, "b": 0}
)

fig_3d.show()

7. Exporting Orbital Data to CSV

The computed trajectory and telemetry variables are organized into a Pandas DataFrame and exported as a CSV file. The dataset includes timestamps, geographic coordinates, altitude, speed, and the propagated TEME position components.

This export makes the results reusable for additional numerical analysis, plotting, validation, or integration into a larger satellite-tracking application.

In [15]:
from google.colab import files

output_file = "mission_control_iss_ground_track.csv"

telemetry.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Row count:", len(telemetry))
display(telemetry.describe().round(3))

files.download(output_file)

Saved: mission_control_iss_ground_track.csv
Row count: 181


,latitude_deg,longitude_deg,altitude_km,speed_km_s,x_teme_km,y_teme_km,z_teme_km
count,181.000,181.000,181.000,181.000,181.000,181.000,181.000
mean,-1.148,-6.449,426.400,7.658,85.237,-108.845,-124.104
std,35.082,106.000,4.236,0.002,3072.897,4787.397,3751.463
min,-51.782,-178.630,419.545,7.654,-4327.552,-6728.065,-5328.429
25%,-34.567,-118.550,422.656,7.656,-3009.236,-4861.196,-3840.620
50%,-1.739,17.211,426.207,7.658,183.231,-241.118,-205.068
75%,31.173,75.686,430.201,7.659,3121.631,4866.998,3501.716
max,51.785,179.142,433.540,7.662,4333.038,6728.047,5326.299


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

8. Mission Control Status Summary

A concise status report is generated from the first sample in the predicted trajectory. It displays the satellite identifier, UTC timestamp, geographic position, altitude, speed, propagation duration, and number of samples.

This final step consolidates the main outputs into a readable operational summary and demonstrates how orbital calculations can support a future Mission Control dashboard.

In [14]:
current = telemetry.iloc[0]

print("=" * 52)
print("        MISSION CONTROL | ISS TRACKING")
print("=" * 52)
print(f"Satellite       : {satellite_name}")
print(f"Timestamp (UTC) : {current.time_utc}")
print(f"Latitude        : {current.latitude_deg:.4f}°")
print(f"Longitude       : {current.longitude_deg:.4f}°")
print(f"Altitude        : {current.altitude_km:.2f} km")
print(f"Speed           : {current.speed_km_s:.3f} km/s")
print(f"Prediction span : {len(telemetry) * 0.5 - 0.5:.1f} minutes")
print(f"Samples         : {len(telemetry)}")
print("Orbit model     : SGP4")
print("Ground coords   : Skyfield / WGS84")
print("=" * 52)


        MISSION CONTROL | ISS TRACKING
Satellite       : ISS (ZARYA)
Timestamp (UTC) : 2026-10-12 11:29:04.771074+00:00
Latitude        : 45.5662°
Longitude       : -60.2561°
Altitude        : 429.29 km
Speed           : 7.657 km/s
Prediction span : 90.0 minutes
Samples         : 181
Orbit model     : SGP4
Ground coords   : Skyfield / WGS84
